In [1]:
import pandas as pd
import numpy as np

In [2]:
f = "../data/precipitacion/B26289.csv"
gasto = pd.read_csv(f, skiprows=7, usecols=[0, 2], index_col=0, parse_dates=[0])
gasto.columns = gasto.columns.str.strip()
gasto

,Gasto(m³/s)
Fecha,
1960-06-01,2.44
1960-06-02,2.03
1960-06-03,1.78
1960-06-04,1.62
1960-06-05,1.58
...,...
2021-06-03,NaN
2021-06-04,NaN
2021-06-05,NaN


In [3]:
gasto["Gasto(m³/s)"] = pd.to_numeric(gasto["Gasto(m³/s)"], errors="coerce")
gasto_mensual = gasto.resample("ME").mean()
gasto_mensual["year"] = gasto_mensual.index.year
gasto_mensual["month"] = gasto_mensual.index.month
gasto_mensual_pivot = gasto_mensual.pivot(index="year", columns="month", values="Gasto(m³/s)")
gasto_mensual_pivot["anual"] = gasto_mensual_pivot.mean(axis=1)
gasto_mensual_pivot.loc["medio_mensual"] = gasto_mensual_pivot.mean(numeric_only=True)
gasto_mensual_pivot.dropna(inplace=True)
gasto_mensual_pivot.to_csv("../data/precipitacion/gasto_mensual.csv")
gasto_mensual_pivot

month,1,2,3,4,5,6,7,8,9,10,11,12,anual
year,,,,,,,,,,,,,
1961,17.692903,11.572143,4.064194,2.311000,1.768387,67.359000,94.618387,91.072581,56.660000,60.543548,40.773000,10.289677,38.227068
1962,5.645161,3.791071,2.817419,22.088000,10.440645,29.723000,47.798065,8.450968,65.645667,19.260645,11.371667,12.027097,19.921617
1963,6.415484,3.629286,2.400645,1.413667,1.876129,11.245000,113.748065,47.123226,30.823667,15.236774,11.319333,6.595484,20.985563
1964,5.548710,3.223448,2.176452,5.427000,2.590968,12.645667,14.572258,2.978065,11.815333,18.507742,14.928000,28.259032,10.222723
1965,10.431290,3.616429,2.622903,7.640667,9.023226,18.698333,54.778387,124.243548,45.956333,30.819677,13.518000,4.765161,27.176163
1966,3.488065,4.292500,3.183548,3.431667,9.904194,102.906667,38.930645,38.600323,66.279667,85.580968,23.313333,7.655484,32.297255
1967,5.818387,4.694286,5.704516,2.792667,4.030323,9.228667,7.771290,60.464194,210.321667,60.235484,26.982000,8.877419,33.910075
1968,6.741613,5.660690,5.495806,6.217333,17.535484,29.089667,62.601935,46.366452,97.773000,56.595806,17.368667,21.765806,31.101022
1969,13.079677,13.185714,9.785806,7.257667,12.718710,3.018000,33.557097,73.766452,256.378333,51.557419,21.541667,13.593226,42.453314


In [4]:
gasto_diseno = pd.DataFrame(gasto_mensual_pivot["anual"].sort_values(ascending=False))
gasto_diseno.index = np.arange(1, len(gasto_diseno) + 1)
gasto_diseno.columns = ["Gasto(m³/s)"]
gasto_diseno["Tr"] = (len(gasto_diseno) + 1) / gasto_diseno.index
gasto_diseno["Pe"] = 1 / gasto_diseno["Tr"]
gasto_diseno

,Gasto(m³/s),Tr,Pe
1,77.721495,55.000000,0.018182
2,75.276412,27.500000,0.036364
3,60.872365,18.333333,0.054545
4,58.810858,13.750000,0.072727
5,53.848557,11.000000,0.090909
6,50.858175,9.166667,0.109091
7,48.768544,7.857143,0.127273
8,48.167420,6.875000,0.145455
9,46.498752,6.111111,0.163636
10,45.936751,5.500000,0.181818


In [5]:
(gasto_diseno.loc[49,"Gasto(m³/s)"] - gasto_diseno.loc[48,"Gasto(m³/s)"]) * (0.85 - gasto_diseno.loc[48,"Pe"]) / (gasto_diseno.loc[49,"Pe"] - gasto_diseno.loc[48,"Pe"]) + gasto_diseno.loc[48,"Gasto(m³/s)"]

np.float64(21.36044985775642)

In [6]:
volumen = gasto * 3600 * 24
volumen.columns = ["Volumen(m³/día)"]
volumen["Volumen(m³/día)"] = pd.to_numeric(volumen["Volumen(m³/día)"], errors="coerce")
volumen_mensual = volumen.resample("ME").sum()
volumen_mensual.columns = ["Volumen (m³/mes)"]
volumen_mensual["year"] = volumen_mensual.index.year
volumen_mensual["month"] = volumen_mensual.index.month
volumen_mensual_pivot = volumen_mensual.pivot(index="year", columns="month", values="Volumen (m³/mes)") 
volumen_mensual_pivot["anual"] = volumen_mensual_pivot.sum(axis=1)
volumen_mensual_pivot.loc["medio_mensual"] = volumen_mensual_pivot.mean(numeric_only=True)
volumen_mensual_pivot.dropna(inplace=True)
volumen_mensual_pivot = volumen_mensual_pivot[(volumen_mensual_pivot != 0).all(axis=1)]
volumen_mensual_pivot.to_csv("../data/precipitacion/volumen_mensual.csv")

In [7]:
Q = (gasto_diseno.loc[49,"Gasto(m³/s)"] - gasto_diseno.loc[48,"Gasto(m³/s)"]) * (0.85 - gasto_diseno.loc[48,"Pe"]) / (gasto_diseno.loc[49,"Pe"] - gasto_diseno.loc[48,"Pe"]) + gasto_diseno.loc[48,"Gasto(m³/s)"]

In [10]:
cargas = [
    [1,Q,1000,0.85,9.81],
    [1,Q/2,1000,0.85,9.81],
    [2,Q,1000,0.85,9.81],
    [2,Q/2,1000,0.85,9.81],
    [3,Q,1000,0.85,9.81],
    [3,Q/2,1000,0.85,9.81],
    [4,Q,1000,0.85,9.81],
    [4,Q/2,1000,0.85,9.81],
    [5,Q,1000,0.85,9.81],
    [5,Q/2,1000,0.85,9.81],
    [10,Q,1000,0.85,9.81],
    [10,Q/2,1000,0.85,9.81],
]

In [12]:
Hb = pd.DataFrame(cargas, columns=["P","Q","gamma","Eta","g"])
Hb

,P,Q,gamma,Eta,g
0,1,21.360450,1000,0.85,9.81
1,1,10.680225,1000,0.85,9.81
2,2,21.360450,1000,0.85,9.81
3,2,10.680225,1000,0.85,9.81
4,3,21.360450,1000,0.85,9.81
5,3,10.680225,1000,0.85,9.81
6,4,21.360450,1000,0.85,9.81
7,4,10.680225,1000,0.85,9.81
8,5,21.360450,1000,0.85,9.81
9,5,10.680225,1000,0.85,9.81


In [15]:
Hb["Hb"] = Hb["P"] * 1e6 / (Hb["gamma"] * Hb["Q"] * Hb["Eta"] * Hb["g"])
Hb

,P,Q,gamma,Eta,g,Hb
0,1,21.360450,1000,0.85,9.81,5.614378
1,1,10.680225,1000,0.85,9.81,11.228757
2,2,21.360450,1000,0.85,9.81,11.228757
3,2,10.680225,1000,0.85,9.81,22.457513
4,3,21.360450,1000,0.85,9.81,16.843135
5,3,10.680225,1000,0.85,9.81,33.686270
6,4,21.360450,1000,0.85,9.81,22.457513
7,4,10.680225,1000,0.85,9.81,44.915026
8,5,21.360450,1000,0.85,9.81,28.071891
9,5,10.680225,1000,0.85,9.81,56.143783
